# DialogSum: GPU pipeline (Colab)

Ноутбук только вызывает скрипты из репозитория: подготовка данных → бейзлайн → zero-shot через vLLM → LoRA (unsloth + trl) → инференс LoRA через vLLM → метрики → LLM-as-judge.

**Перед запуском:** `Runtime → Change runtime type → T4 GPU`. В Secrets (ключик слева) по желанию добавить `WANDB_API_KEY` и `GROQ_API_KEY`. Потом `Runtime → Run all`. Все артефакты сохраняются на Google Drive в `MyDrive/text-generation/outputs`.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf text-generation
!git clone -b feature/lora-summarization https://github.com/semenovfilipp/text-generation.git
%cd /content/text-generation
!mkdir -p /content/drive/MyDrive/text-generation/outputs
!ln -sfn /content/drive/MyDrive/text-generation/outputs outputs
!ls -la

## Окружение

unsloth и vLLM ставим вместе, как в официальных Colab-ноутбуках unsloth. Каждый шаг запускается отдельным процессом (`!python ...`), поэтому после установки runtime перезапускать не нужно, а GPU-память освобождается между шагами.

In [ ]:
!pip install -q unsloth vllm
!pip install -q -r requirements.txt wandb
!pip install -q -e .
!python -c "import torch, vllm, unsloth, trl, peft; print(torch.__version__, vllm.__version__, trl.__version__, peft.__version__)"

In [ ]:
import os
from google.colab import userdata

for name in ['WANDB_API_KEY', 'GROQ_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY', 'HF_TOKEN']:
    try:
        os.environ[name] = userdata.get(name)
        print(name, 'loaded')
    except Exception:
        print(name, 'not set')

## Данные и бейзлайн без ML

In [ ]:
!python scripts/prepare_data.py --config configs/data.yaml
!python scripts/baseline_lead.py --config configs/data.yaml

## Zero-shot Qwen2.5-0.5B-Instruct через vLLM

In [ ]:
!python scripts/infer.py --config configs/infer_zeroshot.yaml

## LoRA-дообучение (unsloth + trl SFTTrainer)

Если unsloth не встанет, запасной вариант: `!python scripts/train_lora.py --config configs/train_lora.yaml --backend peft` (чистые trl + peft).

In [ ]:
!python scripts/train_lora.py --config configs/train_lora.yaml

In [ ]:
from IPython.display import Image, display
import json

display(Image('outputs/models/qwen2.5-0.5b-lora/loss.png'))
print(json.load(open('outputs/models/qwen2.5-0.5b-lora/train_summary.json')))

## Инференс LoRA-модели через vLLM

In [ ]:
!python scripts/infer.py --config configs/infer_lora.yaml

## Метрики

In [ ]:
!python scripts/evaluate.py --output outputs/metrics.csv

In [ ]:
import pandas as pd

from dialogsum.utils import read_jsonl

test = {r['id']: r for r in read_jsonl('data/processed/test.jsonl')}
preds = {n: {p['id']: p['prediction'] for p in read_jsonl(f'outputs/predictions/{n}.jsonl')} for n in ['lead_first_last', 'qwen2.5-0.5b_zeroshot', 'qwen2.5-0.5b_lora']}
for i in ['test_0', 'test_7', 'test_42']:
    print('=' * 100)
    print(test[i]['dialogue'][:800])
    print('\nREF:', test[i]['references'][0])
    for n, p in preds.items():
        print(f'{n}:', p[i])

## LLM-as-judge (опционально, если задан `GROQ_API_KEY`)

Если ключа в Secrets нет, пропусти ячейку: судью можно запустить локально по предсказаниям с Drive.

In [ ]:
if os.environ.get('GROQ_API_KEY'):
    !python scripts/judge.py --config configs/judge.yaml
    !cp -r results/judge outputs/
else:
    print('GROQ_API_KEY not set, skip')

## Сохранить результаты

Скачай `MyDrive/text-generation/colab_results.zip`: там предсказания, метрики, логи и график обучения, без весов.

In [ ]:
!cd outputs && zip -qr /content/drive/MyDrive/text-generation/colab_results.zip predictions metrics.csv models/qwen2.5-0.5b-lora/log_history.json models/qwen2.5-0.5b-lora/loss.png models/qwen2.5-0.5b-lora/train_summary.json $(ls -d judge 2>/dev/null)
!ls -la /content/drive/MyDrive/text-generation/